# RAISE-Vision 1

In [2]:
#RUN THIS
from opentrons import protocol_api
import socket
from opentrons.types import *
import opentrons.execute

protocol = opentrons.execute.get_protocol_api("2.19")
protocol.home()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'
Pol

In [30]:
protocol.home()

In [4]:
#RUN THIS
#SETUP LAYOUT

well = protocol.load_labware("corning_24_wellplate_3.4ml_flat", 8) #corning_96_wellplate_360ul_flat
dryer_tips = protocol.load_labware("opentrons_96_tiprack_20ul", 10)
tips = protocol.load_labware("opentrons_96_tiprack_20ul", 11)
left = protocol.load_instrument("p20_single_gen2", "left", tip_racks=[tips])
right = protocol.load_instrument('p1000_single_gen2', 'right') ##no tipracks for the right

#Drypad
stage = protocol.load_labware("fgl_stage", 4)
drypad = protocol.load_labware("fgl_drypad_tiprack", 6)
trash = protocol.load_labware("fgl_trash", 9)

# drypad = protocol.load_labware("fgl_1000ul_drypad", 3)
# # #ONLY RUN IF USING 1000 uL DP
# drypad.set_offset(x=0,y=0,z=30)

In [5]:
#RUN THIS
#ACTIVATE CAMERA
#FUNCTIONS & CLASSES
class Camera:
    def __init__(self, ip_address: str, port: int):
        self.client_socket = socket.create_connection((ip_address, port))
    
    def take_image(self, fname: str) -> None:
        msg: str = f"take-image,{fname}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
        
    def new_folder(self, dir_name: str) -> None:
        msg: str = f"new-folder,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def analyze_images(self, dir_name: str) -> None:
        msg: str = f"analyze-images,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def __enter__(self):
        return self
    
    def close(self):
        self.client_socket.close()
    
    def __exit__(self, exc_type, exc_val, traceback):
        self.client_socket.close()

def change_offset(x: float = 0.0, y: float = 0.0, z: float = 0.0) -> Location:
        loc = Location(Point(x, y, z), None)
        return loc.move(Point(x, y, z))

def change_drypad_loco(dp_loco, kx, ky) -> Location:
    return dp_loco.move(Point(31.94 *kx, -28.24 * ky, 0))

In [24]:
#RUN THIS
#FUNCTIONS for offset stage
    
# def drySides4p():
#     tips.set_offset(x=0, y=0, z=0)
#     stg = stage.wells()[3]
#     dryer = tips.wells()[0]
#     left.pick_up_tip(dryer)

#         #Right Side
#     stage.set_offset(x=9.5, y=0, z=15)
#     left.move_to(stg.top(30), force_direct = False, speed = 65)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)

#     stage.set_offset(x=6.5, y=0, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)

#     stage.set_offset(x=9.5, y=0, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)
#     left.move_to(stg.top(30), force_direct = False, speed = 35)

#         #Left Side
#     stage.set_offset(x=-10.5, y=0, z=15)
#     left.move_to(stg.top(30), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)

#     stage.set_offset(x=-7.5, y=0, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)

#     stage.set_offset(x=-10.5, y=0, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)
#     left.move_to(stg.top(30), force_direct = False, speed = 35)

#         #Front Side
#     stage.set_offset(x=-1, y=-10, z=15)
#     left.move_to(stg.top(30), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)

#     stage.set_offset(x=-1, y=-7, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)

#     stage.set_offset(x=-1, y=-10, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)
#     left.move_to(stg.top(30), force_direct = False, speed = 35)


#         #Slay Back Side
#     stage.set_offset(x=-1, y=10.5, z=15)
#     left.move_to(stg.top(30), force_direct = False, speed = 35)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)

#     stage.set_offset(x=-1, y=7, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 2)
#     protocol.delay(5)

#     stage.set_offset(x=-1, y=10.5, z=15)
#     left.move_to(stg.top(22), force_direct = False, speed = 35)
#     left.move_to(stg.top(30), force_direct = False, speed = 35)


#     #Throw Away the Tip to Trash place
#     throw = trash.wells()[5]
#     left.drop_tip(throw)

def drySides4p_v2(rolledDryerLoc):
    #tips.set_offset(x=0, y=0, z=0)
    stg = stage.wells()[3]
    dryer = dryer_tips.wells()[rolledDryerLoc] #pickup from defined well
    left.pick_up_tip(dryer)

        #Right Side
    stage.set_offset(x=9.5, y=0, z=15)
    left.move_to(stg.top(30), force_direct = False, speed = 65)
    left.move_to(stg.top(22), force_direct = False, speed = 35)

    stage.set_offset(x=6.5, y=0, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)

    stage.set_offset(x=9.5, y=0, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)

        #Left Side
    stage.set_offset(x=-10.5, y=0, z=15)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 35)

    stage.set_offset(x=-7.5, y=0, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)

    stage.set_offset(x=-10.5, y=0, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)

        #Front Side
    stage.set_offset(x=-1, y=-10, z=15)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 35)

    stage.set_offset(x=-1, y=-7, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)

    stage.set_offset(x=-1, y=-10, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)


        #Slay Back Side
    stage.set_offset(x=-1, y=10.5, z=15)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 35)

    stage.set_offset(x=-1, y=7, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)

    stage.set_offset(x=-1, y=10.5, z=15)
    left.move_to(stg.top(22), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)


    #Throw Away the Tip to Trash place
    throw = trash.wells()[5]
    left.drop_tip(throw)
    
def dryDropletTiprack_20(wellNum): 
    loc = drypad.wells()[wellNum]
    left.pick_up_tip(loc)


    #Dry the actual droplet
    stage.set_offset(x=-0.5, y=0, z=15)
    stg = stage.wells()[3]
    left.move_to(stg.top(100), force_direct = False)
    left.move_to(stg.top(48), force_direct = False, speed = 60)
    left.move_to(stg.top(44), force_direct = False, speed = 2) #43.1
    protocol.delay(3)
    left.move_to(stg.top(100), force_direct = False)

    # #Throw Away the Drypad
    left.drop_tip(loc)
    
    
# def dryDropletTiprack_1000(wellNum): 
#     loc = drypad.wells()[wellNum]
#     right.pick_up_tip(loc)

#     #Dry the actual droplet
#     stage.set_offset(x=-1.5, y=-108.9, z=28)
#     stg = stage.wells()[1]
#     right.move_to(stg.top(100), force_direct = False)
#     right.move_to(stg.top(48), force_direct = False, speed = 60) #48
#     right.move_to(stg.top(44), force_direct = False, speed = 2) #46.1
#     protocol.delay(3)
#     right.move_to(stg.top(100), force_direct = False)
    
#     #Throw Away the Drypad
#     right.drop_tip(loc)

def droplet(j, run_dir):
    stage.set_offset(x=-0.7, y=0.0, z=15)
    stg = stage.wells()[3]

    #Locations 
    locA = stg.top(50)  #(50)
    locB = stg.top(47.4)  #48.75
    
    #ASPIRATE
    pip = tips.wells()[j]
    left.pick_up_tip(pip)

    #prelens image
    fname = run_dir + f"pre_lens_{j}.jpg"
    cam.take_image(fname)

    left.aspirate(5, well["B3"]) # Aspirate 5 uL

    #CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    protocol.delay(1)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)

    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)

    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    fname = run_dir + f"Image_{j}.jpg"
    cam.take_image(fname)
    protocol.delay(1)

    #Drop the tip in the TRASH
    left.drop_tip() 

In [21]:
#Initiate Camera Connection
from time import time
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

In [53]:
a_dry = 0
j_tip = 0

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [55]:
run_dir = '../20260807-senofilcon AOM-4p1m-RV-37 (video_3)/'
cam.new_folder(run_dir)

drySides4p_v2(a_dry)
a_dry += 1
if (a_dry >= 96):
    a_dry = 0

for j in range(10):
    droplet(j_tip, run_dir)
    j_tip += 1
    if (j_tip >= 96):
        j_tip = 0
        
    dryDropletTiprack_20(j)

cam.analyze_images(run_dir)

KeyboardInterrupt: 

In [ ]:
## NEW UPDATED VERSION
#RUN THIS MO
# RAISE VISION, RAISE OT-2 CAMERA CONNECTION - 2
# from time import time
IP_ADDRESS = "192.168.0.111"

PORT = 1200
cam = Camera(IP_ADDRESS, PORT)


try:
    run_dir = '../20260807-senofilcon AOM-4p1m-RV-37 (video_3)/'
    cam.new_folder(run_dir)

    drySides4p_v2(a_dry)
    a_dry += 1
    if (a_dry >= 96):
        a_dry = 0

    for j in range(10):
        droplet(j_tip, run_dir)
        j_tip += 1
        if (j_tip >= 96):
            j_tip = 0

        dryDropletTiprack_20(j)

    cam.analyze_images(run_dir)
        
except:
    pass
finally:
    cam.close()
    left.drop_tip()

In [19]:
#RUN THIS MO
# RAISE VISION, RAISE OT-2 CAMERA CONNECTION - 2
# from time import time
IP_ADDRESS = "192.168.0.111"

PORT = 1200
cam = Camera(IP_ADDRESS, PORT)


try:
    run_dir = '../20260807-comfilcon A-4p1m-RV-37 (5_test)/'
    cam.new_folder(run_dir)
    
    #drySides4p()
    drySides4p_v2(a_dry)
    a_dry += 1
    if (a_dry >= 96):
        a_dry = 0

    for j in range(10):
        droplet(j, run_dir) 
        dryDropletTiprack_20(j)
        
    cam.analyze_images(run_dir)
        
except:
    pass
finally:
    cam.close()
    left.drop_tip()

detach tip called with no tip


In [58]:
protocol.home()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [56]:
left.drop_tip()

<InstrumentContext: p20_single_v2.2 in LEFT>

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

### OTHER

In [140]:
#NEW FUNCTIONS FOR 1000 UL ACCELERATED WORKFLOW & OFFSET STAGE
def drySides4p():
    tips.set_offset(x=0, y=0, z=0)
    stg = stage.wells()[1]
    dryer = tips.wells()[0]
    left.pick_up_tip(dryer)
    
#     #Slay Right Side
    stage.set_offset(x=9, y=-109.5, z=28)
    left.move_to(stg.top(30), force_direct = False, speed = 65)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    
    stage.set_offset(x=6.5, y=-109.5, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=9, y=-109.5, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    
#     #Slay Left Side
    stage.set_offset(x=-11, y=-109.5, z=28)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    
    stage.set_offset(x=-9.5, y=-109.5, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-11, y=-109.5, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    
#     #Slay Front Side
    stage.set_offset(x=-1.5, y=-120, z=28)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    
    stage.set_offset(x=-1.5, y=-118, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.5, y=-120, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    
    
    #Slay Back Side
    stage.set_offset(x=-1.5, y=-100, z=28)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    
    stage.set_offset(x=-1.5, y=-102, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.5, y=-100, z=28)
    left.move_to(stg.top(22.5), force_direct = False, speed = 35)
    left.move_to(stg.top(30), force_direct = False, speed = 35)
    
    
#     #Drop Tip
#     left.drop_tip()
    #Throw Away the Tip to Trash place
    throw = trash.wells()[5]
    left.drop_tip(throw)

def dryDropletTiprack_1000(wellNum): 
    loc = drypad.wells()[wellNum]
    right.pick_up_tip(loc)

    #Dry the actual droplet
    stage.set_offset(x=-1.5, y=-109.5, z=28)
    stg = stage.wells()[1]
    right.move_to(stg.top(100), force_direct = False)
    right.move_to(stg.top(48), force_direct = False, speed = 60) #48
    right.move_to(stg.top(44), force_direct = False, speed = 2) #46.1
    protocol.delay(3)
    right.move_to(stg.top(100), force_direct = False)
    
    #Throw Away the Drypad
    right.drop_tip(loc)
    
def droplet(i, j):
    stage.set_offset(x=-1.6, y=-109.8, z=28)
    stg = stage.wells()[1]
    
    #Locations 
    locA = stg.top(48)
    locB = stg.top(46)  #48.75, 45.9
    
#     #ASPIRATE

#     left.aspirate(5, well["A5"]) # Aspirate 5 uL

#     #CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    protocol.delay(1)
    left.dispense(6, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
#     #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
    fname = f"../Image_{j}_{i}.jpg"
    cam.take_image(fname)

def accelerated_RV(j):
    
    droplet(j)

    #Dry the actual droplet
    stage.set_offset(x=-1.5, y=-109.5, z=28)
    stg = stage.wells()[1]
    right.move_to(stg.top(55), force_direct = False)
    right.move_to(stg.top(48), force_direct = False, speed = 60) #48
    right.move_to(stg.top(44), force_direct = False, speed = 2) #46.1
    protocol.delay(3)
    right.move_to(stg.top(55), force_direct = False)
    
#     #Throw Away the Drypad
#     right.drop_tip(loc)

def accelerated_RV_AG(j):
    loc = drypad.wells()[j]
    right.pick_up_tip(loc)
    
    left.aspirate(5, well["A5"]) # Aspirate 5 uL
    left.air_gap(1)
    left.aspirate(5, well["A5"]) # Aspirate 5 uL
    left.air_gap(1)
    left.aspirate(5, well["A5"]) # Aspirate 5 uL
    left.air_gap(1)
    
    for i in range(3):
        droplet(i, j)

        #Dry the actual droplet
        stage.set_offset(x=-1.5, y=-109.5, z=28)
        stg = stage.wells()[1]
        right.move_to(stg.top(55), force_direct = False)
        right.move_to(stg.top(48), force_direct = False, speed = 60) #48
        right.move_to(stg.top(44), force_direct = False, speed = 2) #46.1
        protocol.delay(3)
        right.move_to(stg.top(55), force_direct = False)
        
    right.drop_tip(loc)
    


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [141]:
#ACCELERATED WORKFLOW
IP_ADDRESS = "192.168.0.111"

PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

try:
#     drySides4p() #takes 52 s for 4 points
    
    fname = f"../pre_lens.jpg"
    cam.take_image(fname)
    
    left.pick_up_tip()
#     loc = drypad.wells()[0]
#     right.pick_up_tip(loc)
    
    protocol.delay(2)


    for j in range(2):
        accelerated_RV_AG(j)
        
    right.drop_tip(loc)
except:
    pass
finally:
    cam.close()
    left.drop_tip()

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 77, in parse_rpm_response
KeyError: 'T'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 144, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 85, in parse_rpm_response
opentrons.drivers.utils.ParseError: ParseError(error_message=Unexpected argument to parse_rpm_response, parse_source=M105 C:370)
Polling exception
Traceback (most recent call last):

In [136]:
# left.pick_up_tip()
# left.aspirate(5, well["A5"]) # Aspirate 5 uL
# left.air_gap(2)
# left.aspirate(5, well["A5"]) # Aspirate 5 uL
# left.air_gap(2)
# left.aspirate(5, well["A5"]) # Aspirate 5 uL

left.dispense(19, well["A6"])

<InstrumentContext: p20_single_v2.2 in LEFT>

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 402, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 415, in read_labware_latch
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 154, in get_labware_latch_status
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: 

In [132]:
right.drop_tip()

<InstrumentContext: p1000_single_v2.2 in RIGHT>

In [6]:
IP_ADDRESS = "192.168.0.111"
PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

fname = f"../910.jpg"
cam.take_image(fname)


cam.close()

In [139]:
left.drop_tip()

<InstrumentContext: p20_single_v2.2 in LEFT>

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

In [20]:
def continuous_droplet(j):
    stage.set_offset(x=-1.5, y=-89.85, z=28)
    stg = stage.wells()[1]
    
    #Locations 
    locA = stg.top(49)
    locB = stg.top(45.9)  #48.75
    

    #ASPIRATE
    left.aspirate(5, well["A5"]) # Aspirate 5 uL

    # CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    protocol.delay(1)
    left.dispense(5, locA, rate = 1) # Dispense 5 uL
    left.move_to(locB, force_direct = True, speed = 0.5)
    
    #Safety Factor
    left.move_to(locA, force_direct = True, speed = 2)
    
    #VIEW DROPLET + TAKE IMAGE   
    protocol.delay(5)
#     fname = f"../Image_{j}.jpg"
#     cam.take_image(fname)
    protocol.delay(1)
    
    
# RAISE VISION, RAISE OT-2 CAMERA CONNECTION - 2
# from time import time
IP_ADDRESS = "192.168.0.111"

PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

try:
    drySides4p() #takes 89 s seconds for 8 points, 55 s for 4 points
    
    
    fname = f"../pre_lens.jpg"
    cam.take_image(fname)
    
    left.pick_up_tip()

    for j in range(10):
        continuous_droplet(j)
        protocol.delay(10)
        
except:
    pass
finally:
    cam.close()
    left.drop_tip()
    


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 120, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response

# RAISE VISION - HEATED LENS DESIGN CODE

In [4]:
#Loading the heating module
hs_mod = protocol.load_module(module_name="heaterShakerModuleV1", location="1")
hs_mod.set_and_wait_for_temperature(55)


Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 63, in parse_temperature_response
KeyError: 'C'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 123, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 65, in parse_temperature_response
opentrons.drivers.utils.ParseError: ParseError(error_message=Unexpected argument to parse_temperature_response, parse_source=IL_LSDO
15C3.5 T:55.00)


ProtocolCommandFailedError: Error 4000 GENERAL_ERROR (ProtocolCommandFailedError): PythonException: opentrons.drivers.utils.ParseError: ParseError(error_message=Unexpected argument to parse_temperature_response, parse_source=IL_LSDO
15C3.5 T:55.00)

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 141, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 399, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 421, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 472, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'
Pol